In [1]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio import AlignIO
import os
import pathlib

In [47]:
os.chdir("/home/kaderi/Feroj/thesis/HRV-C/Data_collecton/hrv-top5/result_collection/VP4_B_CELL")

In [ ]:
alignment_vp1 = AlignIO.read("/home/kaderi/Feroj/thesis/HRV-C/Data_collecton/hrv-top5/C", "fasta")
print(alignment_vp1)
print(alignment_vp1.get_alignment_length())

Alignment with 5 rows and 66 columns
GAQVSKQNVGSHENSVSASGGSVIKYFNINYYKDSASSGLTKQD...ALM HRV-C2_VP4
GAQVSRQSVGSHETMIHAGTGAVVKYFNVNYYKDAASSGLTKQD...ALM HRV-C6_VP4
GAQVSRQKVGSHDNAINASSGSVVKYFNINYYKDAASSGLSKQD...ALM HRV-C9_VP4
GAQVSRQNNGTHESGVTANNGSVIKYFNINYYKDSASSGLSRQD...ALM HRV-C15_VP4
GAQVSKQSVGAHETTVHAGSGAVVKYFNINYYKDAASSGLTKQD...ALM HRV-C43_VP4
66


In [49]:

consensus_sequence = ""
for residue in range(0,alignment_vp1.get_alignment_length()):
    unique_chars = ""
    for i in alignment_vp1[:,residue]:
        if i in unique_chars:
            pass
        else:
            unique_chars+=i
    char_dict = dict()

    for j in unique_chars:

        char_dict.update({f"{j}":alignment_vp1[:,residue].count(j)})


    keys = char_dict.keys()
    if '-' in keys:
        char_dict.pop("-")

    dict_value= list(char_dict.values())

    for keys, values in char_dict.items():
        if values == max(dict_value):
            consensus_sequence += keys
            break

                    

print(consensus_sequence)
print(len(consensus_sequence))
                

GAQVSRQNVGSHENSVHASSGSVVKYFNINYYKDAASSGLTKQDFSQDPSKFTQPVADVLTNPALM
66


In [19]:
# full_conserve_sequence =""
# for i in consensus_sequence:
#     if i != "-":
#         full_conserve_sequence += i
#     else:
#         pass
# full_conserve_sequence

In [38]:
conserved = ""
for i in range(alignment_vp1.get_alignment_length()):
    column = alignment_vp1[:, i]
    if column.count(column[0]) == len(column) and column[0] != "-":
        conserved += column[0]
    else:
        conserved += "-"

print("Conserved pattern:\n", conserved)
print(len(conserved))

Conserved pattern:
 -------------------A-V-S--P-NN----------Y-------------I---PL-M--T-F-TTL-GE-LNY--NWSGS---T--CVC--FSTGKFL-AYTPPG---P--R--AMLG-H--WDLGLQSSC----PW-SS---RRT--------G---LW---
168


In [39]:
## Here we will call the b cell/t-cell eptopes datafram

bpred_result = pd.read_csv(filepath_or_buffer="BPRED_RESULT_VP3.csv")
bpred_result

,Entry,Position,AminoAcid,Exposed/Buried,RelativeSurfaceAccessilibity,HelixProbability,SheetProbability,CoilProbability,EpitopeProbability
0,Sequence,1,Y,E,0.746,0.003,0.003,0.994,0.260333
1,Sequence,2,S,E,0.540,0.018,0.047,0.935,0.325222
2,Sequence,3,P,B,0.255,0.053,0.043,0.903,0.387556
3,Sequence,4,T,E,0.428,0.052,0.084,0.864,0.456222
4,Sequence,5,K,E,0.393,0.052,0.084,0.864,0.509222
...,...,...,...,...,...,...,...,...,...
163,Sequence,164,L,B,0.268,0.021,0.279,0.699,0.536333
164,Sequence,165,W,E,0.340,0.021,0.279,0.699,0.479667
165,Sequence,166,Y,E,0.368,0.019,0.141,0.840,0.421889
166,Sequence,167,Q,E,0.469,0.018,0.047,0.935,0.366889


In [40]:
# for bepripred
epitope_pred = ""

for i,j in enumerate(bpred_result["EpitopeProbability"]):
    if j < 0.5:
        epitope_pred+="-"
    else:
        epitope_pred+=bpred_result['AminoAcid'][i]
epitope_pred

'----KMIHIPG-ITNIMHMARV-------NIPGQVGKVGMYNITVASKTADQERILAIPLQMSNTLFATT--GEILNYYSNWSGSIRLTFMCVCDSFSTGK-------------ETRKEAMLGVHVVWDLGLQSSCTLVVPWISSTFYRRTKADNYTSGGYVTL----'

In [41]:
## now we will cut the epitope separate then
epitope_to_check= []
for i in epitope_pred.split("-"):
    if len(i)>=7:
        epitope_to_check.append(i)
    else:
        pass
epitope_to_check


['KMIHIPG',
 'ITNIMHMARV',
 'NIPGQVGKVGMYNITVASKTADQERILAIPLQMSNTLFATT',
 'GEILNYYSNWSGSIRLTFMCVCDSFSTGK',
 'ETRKEAMLGVHVVWDLGLQSSCTLVVPWISSTFYRRTKADNYTSGGYVTL']

In [42]:
conserve_epitops= []
for i in conserved.split("-"):
    if len(i)>=5:
        conserve_epitops.append(i)
    else:
        pass
conserve_epitops

['NWSGS', 'FSTGKFL', 'AYTPPG', 'WDLGLQSSC']

In [43]:

result_table = pd.DataFrame(columns=("conserve_regions",
                                     'candidate_epitope',
                                     'conservence_presence'))
for a in conserve_epitops:
    l = ''
    k = ''
    for i in range(0,len(a)):

        b = a[i:]
        if b in l:
            break
        elif len(b)>=4:
            for epitopes in epitope_to_check:
                if b in epitopes:
                    print(f"{b} present in {epitopes}")
                    l += b
                    result_table_1 = pd.DataFrame({
                                        "conserve_regions":[a],
                                        "candidate_epitope":[epitopes],
                                        "conservence_presence":[l]})
                    result_table = pd.concat((result_table,result_table_1),axis = 0)
                    break

result_table.describe()

NWSGS present in GEILNYYSNWSGSIRLTFMCVCDSFSTGK
WDLGLQSSC present in ETRKEAMLGVHVVWDLGLQSSCTLVVPWISSTFYRRTKADNYTSGGYVTL


,conserve_regions,candidate_epitope,conservence_presence
count,2,2,2
unique,2,2,2
top,NWSGS,GEILNYYSNWSGSIRLTFMCVCDSFSTGK,NWSGS
freq,1,1,1


In [44]:
conserve_regions_table = []
candidate_epitope_table = []
conserve_epitope_presence = []
for i in result_table["candidate_epitope"].unique():
    index_conserve = result_table[result_table["candidate_epitope"]== i ]['conserve_regions'].values
    presence_conserve = result_table[result_table["candidate_epitope"]== i ]['conservence_presence'].values
    conserve_epitope_region_table = ''
    conserve_epitope_presence_table = ''
    if len(index_conserve)> 1:
        for j in index_conserve:
            conserve_epitope_region_table += f"{j}"
            conserve_epitope_region_table += ","
        for k in presence_conserve:
            conserve_epitope_presence_table += f"{k}"
            conserve_epitope_presence_table += ","
    else:
        conserve_epitope_region_table += f"{index_conserve[0]}"
        conserve_epitope_presence_table += f"{presence_conserve[0]}"
    conserve_regions_table.append(i)
    candidate_epitope_table.append(conserve_epitope_region_table)
    conserve_epitope_presence.append(conserve_epitope_presence_table)    



In [45]:
final_table = pd.DataFrame({"conserve_region":candidate_epitope_table,
                            "epitope":conserve_regions_table,
                            "conservence_presence":conserve_epitope_presence,
                            })
final_table

,conserve_region,epitope,conservence_presence
0,NWSGS,GEILNYYSNWSGSIRLTFMCVCDSFSTGK,NWSGS
1,WDLGLQSSC,ETRKEAMLGVHVVWDLGLQSSCTLVVPWISSTFYRRTKADNYTSGG...,WDLGLQSSC


In [46]:
final_table.to_csv(path_or_buf="VP3_BCELL_final_candidate_epitope.csv")